In [ ]:
"""
=============================================================================
PRE-MODELING 1: persistence of zeros (history rule for the "sin deserción observada" group)
=============================================================================
Question answered: if a sede shows NO dropout in its own history (DESERCION_LAG3 == 0), how often does
it still have a positive dropout rate in the target year? That share is the "false reassurance" rate of
the rule used by the tool to label sedes as "sin deserción observada".

The rule has no parameters (nothing is fitted), so it is evaluated on every eligible row, by target year,
split and segment. Two versions are reported:
  - A: history zero = DESERCION_LAG3 == 0 (includes the imputed zeros, i.e. sedes without Sit_Acad record)
  - B: history zero = DESERCION_LAG3 == 0 and the zero was observed (not imputed)

Input : Data/Processed/panel_final_<panel>.csv (Phase 2 output)
Output: Data/Auditorio_Panel_Final/pre_modelo_persistencia_<panel>.csv and .md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"
PANELES = ["ofrecen11", "Solo_Primaria"]

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE = "SEDE_CODIGO"
COL_ANIO = "ANIO_OBJETIVO"
COL_SEGMENTO = "SEGMENTO_MODELO"
COL_TARGET = "TASA_DESERCION_SEDE"
COL_HIST = "DESERCION_LAG3"
COL_HIST_IMPUTADA = "FLAG_DESERCION_LAG3_CERO_IMPUTADO"
USE_COLS = [
    COL_SEDE, COL_ANIO, "SPLIT", COL_SEGMENTO, "ELEGIBLE_MODELO", "TIENE_HIST_DESERCION",
    COL_HIST, COL_HIST_IMPUTADA, COL_TARGET, "ALTO_RIESGO",
]
VERSIONES = {
    "A_incluye_ceros_imputados": lambda d: d[COL_HIST] == 0,
    "B_solo_ceros_observados": lambda d: (d[COL_HIST] == 0) & (d[COL_HIST_IMPUTADA] == 0),
}

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def summarize(d: pd.DataFrame, in_rule: pd.Series) -> dict:
    """Metrics of the history rule on the rows of d (all of them have history and are eligible)."""
    positive = d[COL_TARGET] > 0
    alto = d["ALTO_RIESGO"].fillna(0).astype(int) == 1
    n, n_rule = len(d), int(in_rule.sum())
    out_rule = ~in_rule
    return {
        "FILAS": n,
        "FILAS_EN_REGLA": n_rule,
        "PCT_EN_REGLA": 100 * n_rule / n if n else np.nan,
        # Share of the rule group that still shows dropout: the false reassurance of the tool
        "PCT_FALSA_TRANQUILIDAD": 100 * float(positive[in_rule].mean()) if n_rule else np.nan,
        # Same share outside the rule, to read how much the history separates the two groups
        "PCT_POSITIVO_FUERA_REGLA": 100 * float(positive[out_rule].mean()) if out_rule.any() else np.nan,
        # Positives and PRIORITARIO sedes that the rule would label as "sin deserción observada"
        "PCT_POSITIVOS_EN_REGLA": 100 * float((positive & in_rule).sum() / positive.sum()) if positive.sum() else np.nan,
        "PCT_PRIORITARIO_EN_REGLA": 100 * float((alto & in_rule).sum() / alto.sum()) if alto.sum() else np.nan,
    }


def evaluate_panel(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Evaluate both versions of the rule by target year, split and segment."""
    base = df[(df["ELEGIBLE_MODELO"] == 1) & (df["TIENE_HIST_DESERCION"] == 1)].copy()
    filas = []
    for version, regla in VERSIONES.items():
        in_rule = regla(base)
        cortes = [
            ("TOTAL", lambda d: pd.Series("todas", index=d.index)),
            ("ANIO_OBJETIVO", lambda d: d[COL_ANIO].astype(int).astype(str)),
            ("SPLIT", lambda d: d["SPLIT"]),
            ("SEGMENTO", lambda d: d[COL_SEGMENTO]),
        ]
        for tipo, clave in cortes:
            grupos = clave(base)
            for g, idx in base.groupby(grupos).groups.items():
                fila = {"PANEL": nombre, "VERSION_REGLA": version, "CORTE": tipo, "GRUPO": g}
                fila.update(summarize(base.loc[idx], in_rule.loc[idx]))
                filas.append(fila)
    return pd.DataFrame(filas)


def to_markdown(df: pd.DataFrame) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla: pd.DataFrame, nombre: str) -> None:
    """Write csv and md; a locked OneDrive file must not abort the analysis."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(AUDIT_DIR / f"pre_modelo_persistencia_{nombre}.csv", index=False)
        resumen = tabla[tabla["CORTE"].isin(["TOTAL", "ANIO_OBJETIVO", "SEGMENTO"])]
        md = [f"# Persistencia de ceros - panel {nombre}", "", to_markdown(resumen)]
        (AUDIT_DIR / f"pre_modelo_persistencia_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre in PANELES:
        path = IN_DIR / f"panel_final_{nombre}.csv"
        df = pd.read_csv(path, usecols=USE_COLS, dtype={COL_SEDE: str}, low_memory=False)
        tabla = evaluate_panel(df, nombre)
        save_report(tabla, nombre)

        print(f"\n=== {nombre}: false reassurance of the history rule (eligible rows with history) ===")
        vista = tabla[(tabla["CORTE"].isin(["TOTAL", "ANIO_OBJETIVO"]))][
            ["VERSION_REGLA", "CORTE", "GRUPO", "FILAS", "PCT_EN_REGLA", "PCT_FALSA_TRANQUILIDAD",
             "PCT_POSITIVO_FUERA_REGLA", "PCT_POSITIVOS_EN_REGLA", "PCT_PRIORITARIO_EN_REGLA"]
        ]
        print(vista.round(1).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 2: how to treat the size of the sede in the dropout target
=============================================================================
Compares several ways of predicting dropout while handling the small-sede problem (a rate of 1 / n for
tiny sedes), with the SAME learner and the SAME temporal backtests, so that only the treatment changes.

Methods (all produce a predicted dropout RATE for each sede):
  B0_historia      baseline: the rate of the sede's own history (DESERCION_LAG3)
  B1_tamano        baseline: every sede gets the same rate, so the ranking is driven by size alone
  M1_pos_tasa      only sedes with a positive rate; target = log(rate); weight = students
  M2_pos_K<k>      same, but the rate is shrunk toward the mean with K pseudo-students (K = 10, 50, 200 and the
                   empirical-Bayes K estimated from the training years)
  M3_dos_partes    probability of a positive rate x the rate of M1 (two-part model)
  M4_poisson       all sedes with a record; Poisson loss on the rate with students as weight (equivalent to
                   modeling the number of deserters with the students as exposure)

Priority of a sede = predicted rate x students ^ ALPHA. ALPHA = 1 is the expected number of students at risk,
ALPHA = 0 is the pure rate; the values in between protect the small sedes without ignoring the size.

Backtests: for each test year, train on the earlier target years only. Population: eligible sedes with a
Sit_Acad record (students known); imputed zeros are "sin dato" and stay out.

Input : Data/Processed/panel_final_<panel>.csv (needs the column PESO_ESTUDIANTES)
Output: Data/Auditorio_Panel_Final/pre_modelo_experimento_tamano_<panel>.csv and .md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO = "SEDE_CODIGO", "ANIO_OBJETIVO"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}

TEST_YEARS = [2021, 2022, 2023]     # Each one is predicted with the earlier target years only
K_GRID = [10.0, 50.0, 200.0]        # Pseudo-students of the shrinkage; the empirical-Bayes K is added
ALPHAS = [0.0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 0.75, 1.0]  # Priority = rate x students ^ ALPHA
TOP_SHARES = [0.10, 0.20]           # Share of sedes in the priority list
HGB_PARAMS = dict(
    learning_rate=0.05, max_iter=300, max_depth=4, min_samples_leaf=50,
    l2_regularization=1.0, random_state=42,
)
SEED = 42

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """
    Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial) by the method of moments
    (Kleinman, 1973). K is the number of pseudo-students that the shrinkage adds: small K means sedes truly
    differ from each other, large K means most of the differences between small sedes are noise.
    """
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = (s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n)
    rho = float(np.clip(rho, 1e-6, 0.99))
    return (1 - rho) / rho


def shrunk_rate(deserters, students, mean_rate, k) -> np.ndarray:
    """Posterior mean of the rate: (deserters + K * mean) / (students + K)."""
    return (np.asarray(deserters, float) + k * mean_rate) / (np.asarray(students, float) + k)


def load_population(nombre: str) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    df = pd.read_csv(IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str}, low_memory=False)
    if COL_PESO not in df.columns:
        raise KeyError(f"{COL_PESO} is missing: rerun the master panel and Phase 2 with the denominator restored.")
    df = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    df["_d"] = np.round(df[COL_RATE] * df[COL_PESO])
    return df


def fit_predict(df_train: pd.DataFrame, df_test: pd.DataFrame, features: list, k_eb: float) -> dict:
    """Train every method on df_train and return {method: predicted rate for df_test}."""
    pos = df_train[df_train[COL_RATE] > 0]
    m_all = df_train["_d"].sum() / df_train[COL_PESO].sum()
    X_pos, X_all, X_test = pos[features], df_train[features], df_test[features]
    out = {}

    out["B0_historia"] = df_test[COL_HIST].fillna(0.0).to_numpy()
    out["B1_tamano"] = np.full(len(df_test), m_all)

    # M1: positives only, log rate, students as weight (conditional rate: it assumes the sede has dropout)
    sev = HistGradientBoostingRegressor(loss="squared_error", **HGB_PARAMS)
    sev.fit(X_pos, np.log(pos[COL_RATE]), sample_weight=pos[COL_PESO])
    sev_pred = np.exp(sev.predict(X_test))
    out["M1_pos_tasa"] = sev_pred

    # M2: positives only, log of the shrunk rate for each K
    for k in K_GRID + ([k_eb] if np.isfinite(k_eb) else []):
        y = shrunk_rate(pos["_d"], pos[COL_PESO], m_all, k)
        mdl = HistGradientBoostingRegressor(loss="squared_error", **HGB_PARAMS)
        mdl.fit(X_pos, np.log(y), sample_weight=pos[COL_PESO])
        etiqueta = "M2_pos_K_EB" if k == k_eb else f"M2_pos_K{k:.0f}"  # Stable label; the K used goes in K_USADO
        out[etiqueta] = np.exp(mdl.predict(X_test))

    # M3: two parts = P(rate > 0) x severity (the first part is only a component of the expected value)
    clf = HistGradientBoostingClassifier(**HGB_PARAMS)
    clf.fit(X_all, (df_train[COL_RATE] > 0).astype(int))
    out["M3_dos_partes"] = clf.predict_proba(X_test)[:, 1] * sev_pred

    # M4: Poisson loss on the rate with students as weight, all sedes with a record
    poi = HistGradientBoostingRegressor(loss="poisson", **HGB_PARAMS)
    poi.fit(X_all, df_train[COL_RATE], sample_weight=df_train[COL_PESO])
    out["M4_poisson"] = poi.predict(X_test)
    return out


def poisson_deviance(d: np.ndarray, mu: np.ndarray) -> float:
    """Mean Poisson deviance of the expected number of deserters (lower is better)."""
    mu = np.clip(mu, 1e-9, None)
    term = np.where(d > 0, d * np.log(np.where(d > 0, d, 1.0) / mu), 0.0)
    return float(np.mean(2 * (term - (d - mu))))


def within_stratum_capture(d: np.ndarray, rate_hat: np.ndarray, quintil: np.ndarray, rng, share: float = 0.10) -> list:
    """
    Risk discrimination NET of size: inside each size quintile, share of the stratum's deserters that fall in
    the top `share` of its sedes ranked by predicted rate. A random list gets `share` (10%).
    """
    caps = []
    for qn in range(5):
        idx = np.where(quintil == qn)[0]
        if len(idx) == 0 or d[idx].sum() == 0:
            caps.append(np.nan)
            continue
        k = int(np.ceil(share * len(idx)))
        jitter = rng.random(len(idx)) * 1e-12 * max(float(np.nanmax(rate_hat[idx])), 1e-12)
        top = idx[np.argsort(-(rate_hat[idx] + jitter))[:k]]
        caps.append(100 * d[top].sum() / d[idx].sum())
    return caps


def evaluate(df_test: pd.DataFrame, rate_hat: np.ndarray, metodo: str, anio: int, panel: str, k_usado=np.nan) -> list:
    """Capture of deserters and fairness to small sedes for each ALPHA and each priority-list size."""
    rng = np.random.default_rng(SEED)
    d = df_test["_d"].to_numpy()
    n_pred = df_test[COL_SIZE].fillna(df_test[COL_SIZE].median()).clip(lower=1).to_numpy()
    quintil = pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy()  # 0 = smallest
    mu = rate_hat * df_test[COL_PESO].to_numpy()
    calib = float(mu.sum() / d.sum()) if d.sum() else np.nan
    dev = poisson_deviance(d, mu)
    intra = within_stratum_capture(d, rate_hat, quintil, rng)

    filas = []
    for alpha in ALPHAS:
        score = rate_hat * n_pred ** alpha
        score = score + rng.random(len(score)) * 1e-12 * max(float(np.nanmax(score)), 1e-12)  # random tie-break
        orden = np.argsort(-score)
        rho = spearmanr(score, d)[0]
        for q in TOP_SHARES:
            k = int(np.ceil(q * len(d)))
            top = np.zeros(len(d), bool)
            top[orden[:k]] = True
            captura = d[top].sum() / d.sum() if d.sum() else np.nan
            fila = {
                "PANEL": panel, "ANIO_TEST": anio, "METODO": metodo, "K_USADO": k_usado, "ALPHA": alpha, "TOP": q,
                "CAPTURA_DESERTORES": 100 * captura,
                "LIFT": captura / q if np.isfinite(captura) else np.nan,
                "PCT_TOP_EN_Q1_PEQUENAS": 100 * float((top & (quintil == 0)).sum() / k),
                "CAPTURA_Q1_PEQUENAS": 100 * float(d[top & (quintil == 0)].sum() / max(d[quintil == 0].sum(), 1)),
                "CAPTURA_Q5_GRANDES": 100 * float(d[top & (quintil == 4)].sum() / max(d[quintil == 4].sum(), 1)),
                "SPEARMAN": rho, "CALIBRACION_ESPERADO_SOBRE_OBSERVADO": calib, "DEVIANZA_POISSON": dev,
                "CAPTURA_DENTRO_ESTRATO_PROM": float(np.nanmean(intra)),
                "CAPTURA_DENTRO_Q1": intra[0], "CAPTURA_DENTRO_Q5": intra[4],
            }
            filas.append(fila)
    return filas


def to_markdown(df: pd.DataFrame) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla: pd.DataFrame, nombre: str) -> None:
    """Write csv and md; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(AUDIT_DIR / f"pre_modelo_experimento_tamano_{nombre}.csv", index=False)
        base = tabla[tabla["TOP"] == 0.10]
        resumen = (
            base.groupby(["METODO", "ALPHA"])[
                ["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "PCT_TOP_EN_Q1_PEQUENAS", "SPEARMAN",
                 "CALIBRACION_ESPERADO_SOBRE_OBSERVADO", "DEVIANZA_POISSON"]
            ].mean().reset_index()
        )
        md = [
            f"# Experimento de tratamiento del tamaño - panel {nombre}",
            "\n## Promedio de los años de prueba, lista del 10% de sedes (CAPTURA = % de los desertores del año que quedan en la lista)\n",
            to_markdown(resumen.sort_values(["ALPHA", "CAPTURA_DESERTORES"], ascending=[True, False])),
            "\n## Discriminación de riesgo dentro de cada estrato de tamaño (promedio de los años; una lista al azar captura 10%)\n",
            to_markdown(
                base[base["ALPHA"] == ALPHAS[0]].groupby("METODO")[
                    ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5"]
                ].mean().reset_index().sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False)
            ),
            "\n## Detalle por año de prueba\n",
            to_markdown(base.drop(columns=["PANEL", "TOP"])),
        ]
        (AUDIT_DIR / f"pre_modelo_experimento_tamano_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def run_panel(nombre: str, features: list) -> pd.DataFrame:
    """Rolling-origin backtests for one panel."""
    df = load_population(nombre)
    filas = []
    for anio in TEST_YEARS:
        train, test = df[df[COL_ANIO] < anio], df[df[COL_ANIO] == anio]
        if train.empty or test.empty:
            print(f"[WARN] {nombre} {anio}: no train or test rows, skipped.")
            continue
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        print(f"{nombre} | test {anio} | train rows {len(train):,} | test rows {len(test):,} | K empirical Bayes = {k_eb:.1f}")
        predicciones = fit_predict(train, test, features, k_eb)
        k_por_metodo = {f"M2_pos_K{k:.0f}": k for k in K_GRID}
        k_por_metodo["M2_pos_K_EB"] = k_eb
        for metodo, rate_hat in predicciones.items():
            filas.extend(evaluate(test, rate_hat, metodo, anio, nombre, k_por_metodo.get(metodo, np.nan)))
    return pd.DataFrame(filas)


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre, features in PANELES.items():
        tabla = run_panel(nombre, features)
        save_report(tabla, nombre)
        top = tabla[tabla["TOP"] == 0.10]
        print(f"\n=== {nombre}: mean over the test years, list of 10% of the sedes ===")
        print(
            top.groupby(["METODO", "ALPHA"])[["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "CALIBRACION_ESPERADO_SOBRE_OBSERVADO"]]
            .mean().round(2).to_string()
        )


if __name__ == "__main__":
    main()


In [ ]:
"""
=============================================================================
PRE-MODELING 3: does a better use of the sede's own history improve the Poisson model?
=============================================================================
Finding of the size experiment: almost all the risk signal is the sede's own history (DESERCION_LAG3), and
the model barely beats it inside each size stratum (it is weak for the large sedes of Solo_Primaria).
Hypothesis: the raw rate of t-3 is very noisy for small sedes and the model cannot tell how reliable it is.

Feature sets compared with the SAME learner (Poisson, students as weight) and the SAME backtests:
  BASE              current predictors (the history enters only as the raw rate DESERCION_LAG3)
  AJUSTADA          + DESERCION_LAG3_AJUSTADA = (deserters + K * mean) / (students + K), K = empirical Bayes
  CONTEOS           + ESTUDIANTES_LAG3 and DESERTORES_LAG3 (students and deserters of the sede in t-3)
  AJUSTADA+CONTEOS  all of them
Baselines without a model: raw history, shrunk history and size only.

The t-3 counts are rebuilt from the final panel itself by calendar year: the row of the same sede whose
ANIO_OBJETIVO is t-3 carries its students (PESO_ESTUDIANTES) and its rate. Sedes without a Sit_Acad record in
t-3 have no counts (NaN), never zero.

Input : Data/Processed/panel_final_<panel>.csv (needs PESO_ESTUDIANTES)
Output: Data/Auditorio_Panel_Final/pre_modelo_experimento_historia_<panel>.csv and .md
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO = "SEDE_CODIGO", "ANIO_OBJETIVO"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"
HIST_LAG = 3

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}
CONJUNTOS = {  # name -> extra history features added to the base predictors
    "BASE": [],
    "AJUSTADA": ["DESERCION_LAG3_AJUSTADA"],
    "CONTEOS": ["ESTUDIANTES_LAG3", "DESERTORES_LAG3"],
    "AJUSTADA+CONTEOS": ["DESERCION_LAG3_AJUSTADA", "ESTUDIANTES_LAG3", "DESERTORES_LAG3"],
}

TEST_YEARS = [2021, 2022, 2023]
ALPHAS = [0.0, 0.1, 0.2, 0.3, 1.0]   # Priority = rate x students ^ ALPHA
TOP_SHARE = 0.10
HGB_PARAMS = dict(
    learning_rate=0.05, max_iter=300, max_depth=4, min_samples_leaf=50,
    l2_regularization=1.0, random_state=42,
)
SEED = 42

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial), method of moments (Kleinman, 1973)."""
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = float(np.clip((s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n), 1e-6, 0.99))
    return (1 - rho) / rho


def load_panel(nombre: str) -> pd.DataFrame:
    """Final panel plus the students and deserters of the same sede in t-3, joined by calendar year."""
    df = pd.read_csv(IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str}, low_memory=False)
    if COL_PESO not in df.columns:
        raise KeyError(f"{COL_PESO} is missing: rerun the master panel and Phase 2 with the denominator restored.")
    hist = df[[COL_SEDE, COL_ANIO, COL_RATE, COL_PESO]].copy()
    hist[COL_ANIO] = hist[COL_ANIO] + HIST_LAG
    hist = hist.rename(columns={COL_RATE: "_rate_lag3", COL_PESO: "ESTUDIANTES_LAG3"})
    df = df.merge(hist, on=[COL_SEDE, COL_ANIO], how="left", validate="many_to_one")
    df["DESERTORES_LAG3"] = np.round(df["_rate_lag3"] * df["ESTUDIANTES_LAG3"])  # NaN where there is no record

    # The rebuilt history must match the DESERCION_LAG3 column of the panel
    both = df["_rate_lag3"].notna() & df[COL_HIST].notna()
    if both.any() and not np.allclose(df.loc[both, "_rate_lag3"], df.loc[both, COL_HIST], atol=1e-9):
        raise ValueError("The rate rebuilt for t-3 does not match DESERCION_LAG3: check the panel.")
    return df


def population(df: pd.DataFrame) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    pop = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    pop["_d"] = np.round(pop[COL_RATE] * pop[COL_PESO])
    return pop


def add_shrunk_history(train: pd.DataFrame, test: pd.DataFrame, k: float) -> tuple:
    """
    DESERCION_LAG3_AJUSTADA = (deserters_t-3 + K * mean) / (students_t-3 + K). K and the mean come from the TRAIN
    years only (two global constants); a sede without counts in t-3 gets the prior mean.
    """
    m_all = train["_d"].sum() / train[COL_PESO].sum()
    out = []
    for frame in (train, test):
        frame = frame.copy()
        known = frame["ESTUDIANTES_LAG3"].gt(0)
        shrunk = (frame["DESERTORES_LAG3"] + k * m_all) / (frame["ESTUDIANTES_LAG3"] + k)
        frame["DESERCION_LAG3_AJUSTADA"] = np.where(known, shrunk, m_all)
        out.append(frame)
    return out[0], out[1]


def poisson_deviance(d: np.ndarray, mu: np.ndarray) -> float:
    """Mean Poisson deviance of the expected number of deserters (lower is better)."""
    mu = np.clip(mu, 1e-9, None)
    term = np.where(d > 0, d * np.log(np.where(d > 0, d, 1.0) / mu), 0.0)
    return float(np.mean(2 * (term - (d - mu))))


def within_stratum_capture(d, rate_hat, quintil, rng, share=0.10) -> list:
    """Inside each size quintile, % of the stratum's deserters in the top `share` of its sedes by predicted rate (random = 10%)."""
    caps = []
    for qn in range(5):
        idx = np.where(quintil == qn)[0]
        if len(idx) == 0 or d[idx].sum() == 0:
            caps.append(np.nan)
            continue
        k = int(np.ceil(share * len(idx)))
        jitter = rng.random(len(idx)) * 1e-12 * max(float(np.nanmax(rate_hat[idx])), 1e-12)
        top = idx[np.argsort(-(rate_hat[idx] + jitter))[:k]]
        caps.append(100 * d[top].sum() / d[idx].sum())
    return caps


def evaluate(df_test, rate_hat, conjunto, anio, panel) -> list:
    """Capture of deserters, fairness to small sedes and risk discrimination net of size."""
    rng = np.random.default_rng(SEED)
    d = df_test["_d"].to_numpy()
    n_pred = df_test[COL_SIZE].fillna(df_test[COL_SIZE].median()).clip(lower=1).to_numpy()
    quintil = pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy()
    mu = rate_hat * df_test[COL_PESO].to_numpy()
    intra = within_stratum_capture(d, rate_hat, quintil, rng)
    filas = []
    for alpha in ALPHAS:
        score = rate_hat * n_pred ** alpha
        score = score + rng.random(len(score)) * 1e-12 * max(float(np.nanmax(score)), 1e-12)
        top = np.zeros(len(d), bool)
        top[np.argsort(-score)[: int(np.ceil(TOP_SHARE * len(d)))]] = True
        filas.append({
            "PANEL": panel, "ANIO_TEST": anio, "CONJUNTO": conjunto, "ALPHA": alpha,
            "CAPTURA_DESERTORES": 100 * d[top].sum() / d.sum() if d.sum() else np.nan,
            "CAPTURA_Q1_PEQUENAS": 100 * float(d[top & (quintil == 0)].sum() / max(d[quintil == 0].sum(), 1)),
            "SPEARMAN": spearmanr(score, d)[0],
            "CALIBRACION": float(mu.sum() / d.sum()) if d.sum() else np.nan,
            "DEVIANZA_POISSON": poisson_deviance(d, mu),
            "CAPTURA_DENTRO_ESTRATO_PROM": float(np.nanmean(intra)),
            "CAPTURA_DENTRO_Q1": intra[0], "CAPTURA_DENTRO_Q5": intra[4],
        })
    return filas


def to_markdown(df: pd.DataFrame) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.2f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla: pd.DataFrame, nombre: str) -> None:
    """Write csv and md; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(AUDIT_DIR / f"pre_modelo_experimento_historia_{nombre}.csv", index=False)
        medias = tabla.groupby(["CONJUNTO", "ALPHA"])[
            ["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "SPEARMAN", "CALIBRACION", "DEVIANZA_POISSON"]
        ].mean().reset_index()
        intra = tabla[tabla["ALPHA"] == ALPHAS[0]].groupby("CONJUNTO")[
            ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON"]
        ].mean().reset_index().sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False)
        # Consistency across years: difference with BASE in the discrimination net of size
        por_anio = tabla[tabla["ALPHA"] == ALPHAS[0]].pivot_table(
            index="CONJUNTO", columns="ANIO_TEST", values="CAPTURA_DENTRO_ESTRATO_PROM"
        )
        ref = "M4_BASE"  # The model labels carry the prefix M4_ (the previous version looked for "BASE" and printed raw values)
        dif = por_anio.sub(por_anio.loc[ref], axis=1).reset_index() if ref in por_anio.index else por_anio.reset_index()
        dif.columns = [str(c) for c in dif.columns]
        md = [
            f"# Experimento de historia - panel {nombre}",
            "\n## Discriminación de riesgo dentro de cada estrato de tamaño (promedio de los años; azar = 10%)\n",
            to_markdown(intra),
            "\n## Diferencia frente a M4_BASE por año de prueba (puntos de captura dentro del estrato; positivo = mejor que BASE)\n",
            to_markdown(dif),
            "\n## Lista del 10% de sedes por conjunto y alpha (promedio de los años)\n",
            to_markdown(medias),
        ]
        (AUDIT_DIR / f"pre_modelo_experimento_historia_{nombre}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def run_panel(nombre: str, base_features: list) -> pd.DataFrame:
    """Rolling-origin backtests for one panel: one Poisson model per feature set plus the baselines."""
    pop = population(load_panel(nombre))
    filas = []
    for anio in TEST_YEARS:
        train, test = pop[pop[COL_ANIO] < anio], pop[pop[COL_ANIO] == anio]
        if train.empty or test.empty:
            print(f"[WARN] {nombre} {anio}: no train or test rows, skipped.")
            continue
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        print(f"{nombre} | test {anio} | train rows {len(train):,} | test rows {len(test):,} | K empirical Bayes = {k_eb:.1f}")
        train, test = add_shrunk_history(train, test, k_eb)

        # Baselines without a model
        filas.extend(evaluate(test, test[COL_HIST].fillna(0.0).to_numpy(), "B0_historia_cruda", anio, nombre))
        filas.extend(evaluate(test, test["DESERCION_LAG3_AJUSTADA"].to_numpy(), "B0b_historia_ajustada", anio, nombre))
        filas.extend(evaluate(test, np.full(len(test), train["_d"].sum() / train[COL_PESO].sum()), "B1_tamano", anio, nombre))

        # Poisson with students as weight, one model per feature set
        for conjunto, extra in CONJUNTOS.items():
            features = base_features + extra
            mdl = HistGradientBoostingRegressor(loss="poisson", **HGB_PARAMS)
            mdl.fit(train[features], train[COL_RATE], sample_weight=train[COL_PESO])
            filas.extend(evaluate(test, mdl.predict(test[features]), f"M4_{conjunto}", anio, nombre))
    return pd.DataFrame(filas)


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre, features in PANELES.items():
        tabla = run_panel(nombre, features)
        save_report(tabla, nombre)
        print(f"\n=== {nombre}: risk discrimination net of size (mean over test years; random = 10%) ===")
        print(
            tabla[tabla["ALPHA"] == ALPHAS[0]].groupby("CONJUNTO")[
                ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON"]
            ].mean().round(2).sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False).to_string()
        )


if __name__ == "__main__":
    main()


In [2]:
"""
=============================================================================
PRE-MODELING 4: the sede's own history as the BASE of the model (offset) vs as one more feature
=============================================================================
Finding of the history experiment: the shrunk history alone (no model) discriminates as well as any variant of
the model inside each size stratum, and a SHAP built on top of it would be dominated by DESERCION_LAG3, which is
not actionable. Idea: let the history set the baseline of each sede and let the model learn only the deviation
from that baseline with the remaining (actionable) predictors. Then SHAP can explain "everything except the lag".

Variants (all Poisson, students as weight, same backtests 2021-2023, same LightGBM settings):
  A    history as FEATURES (raw rate + shrunk rate + t-3 counts), no offset. Strongest feature-based comparator.
  B    OFFSET = Poisson GLM on log(shrunk history) with a FREE slope; the GBM sees only the NON-history
       predictors. This is the SHAP-friendly design (history lives in the offset, outside the explained part).
  B2   same offset, but the GBM also sees the history features (cost of leaving the lag out of the GBM = B vs B2).
  C    B2 + monotone increasing constraint on the two history rates.
  Each one is run with two capacities (number of trees).
Baselines: GLM_solo (the offset model alone), shrunk history without model, and the previous HistGradientBoosting
(M4_BASE_HGB) to check that the earlier numbers are reproduced.

Offset in LightGBM: the GLM linear predictor is passed as init_score at fit time and added back by hand at
prediction time (predict(raw_score=True) does not include it). Students enter as sample weight and the target is
the rate, which is the same as modelling deserters with log(students) as offset.

Uncertainty: paired bootstrap by MUNICIPIO (the same resampled municipalities are used in every test year and
every variant), 95% percentile interval of the difference against the reference. Differences of about one point
are only trusted if the interval excludes zero.

SHAP by family (last test year): LightGBM TreeSHAP (pred_contrib). In B the history is the offset, so its share is
the mean absolute deviation of the GLM linear predictor; the table also shows the shares WITHOUT the history.

Input : Data/Processed/panel_final_<panel>.csv (needs PESO_ESTUDIANTES and COD_MPIO_DANE)
Output: Data/Auditorio_Panel_Final/pre_modelo_experimento_offset_<panel>.csv and .md
        Data/Auditorio_Panel_Final/pre_modelo_experimento_offset_bootstrap_<panel>.csv
        Data/Auditorio_Panel_Final/pre_modelo_experimento_offset_shap_familias_<panel>.csv
=============================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor

try:
    import lightgbm as lgb
except ImportError as err:  # corporate machines sometimes block pip: fail early with a clear message
    raise ImportError("lightgbm is required for this experiment (pip install lightgbm).") from err

# =============================================================================
# 0. PATHS
# =============================================================================
BASE_DIR = Path(
    r"C:\Users\DELL\OneDrive\Escritorio\UNIVERSIDAD\Maestria Business A\Proyecto Empresarial\Data"
)
IN_DIR = BASE_DIR / "Processed"
AUDIT_DIR = BASE_DIR / "Auditorio_Panel_Final"

# =============================================================================
# 1. GENERAL VARIABLES
# =============================================================================
COL_SEDE, COL_ANIO, COL_MPIO = "SEDE_CODIGO", "ANIO_OBJETIVO", "COD_MPIO_DANE"
COL_RATE, COL_PESO, COL_SIZE = "TASA_DESERCION_SEDE", "PESO_ESTUDIANTES", "MATRICULA_TOTAL"
COL_HIST = "DESERCION_LAG3"
COL_HIST_ADJ = "DESERCION_LAG3_AJUSTADA"
COLS_HIST_COUNTS = ["ESTUDIANTES_LAG3", "DESERTORES_LAG3"]
COLS_HIST_ALL = [COL_HIST, COL_HIST_ADJ] + COLS_HIST_COUNTS
HIST_LAG = 3

PREDICTORES_COMUNES = [
    "DESERCION_LAG3", "MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO", "PROP_DESPLAZADOS",
    "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE", "FLAG_PDET", "FLAG_ZOMAC",
]
PREDICTORES_ICFES = ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"]
PANELES = {
    "ofrecen11": PREDICTORES_COMUNES + PREDICTORES_ICFES,
    "Solo_Primaria": PREDICTORES_COMUNES,
}
FAMILIAS = {  # intervention-oriented families used for the SHAP shares
    "Historia propia": COLS_HIST_ALL,
    "Tamaño y dinámica de matrícula": ["MATRICULA_TOTAL", "MATRICULA_PCT_CAMBIO"],
    "Composición del alumnado": ["PROP_DESPLAZADOS", "PROP_LIMITACION", "PROP_ETNIA", "PROP_FLEXIBLE"],
    "Contexto territorial": ["FLAG_PDET", "FLAG_ZOMAC"],
    "ICFES": ["ICFES_PROM_PUNT_GLOBAL", "ICFES_PROM_INSE"],
}

TEST_YEARS = [2021, 2022, 2023]
ALPHAS = [0.0, 0.1, 0.2, 0.3, 1.0]   # Priority = rate x students ^ ALPHA
TOP_SHARE = 0.10
TREE_GRID = [150, 300]               # two capacities; the last one is the reference capacity
N_BOOT = 200                         # municipality bootstrap replicates (lower it for a quick run)
SHAP_MAX_ROWS = 20000
SEED = 42

HGB_PARAMS = dict(                   # identical to the previous experiment (reproduction check)
    learning_rate=0.05, max_iter=300, max_depth=4, min_samples_leaf=50,
    l2_regularization=1.0, random_state=42,
)
LGBM_PARAMS = dict(
    objective="poisson", learning_rate=0.05, num_leaves=15, max_depth=4, min_child_samples=50,
    reg_lambda=1.0, random_state=SEED, n_jobs=4, verbose=-1,
)
METRICS_BOOT = ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON"]
LOWER_IS_BETTER = {"DEVIANZA_POISSON"}

# =============================================================================
# 2. FUNCTION DEFINITIONS
# =============================================================================

def estimate_k_empirical_bayes(deserters, students) -> float:
    """Empirical-Bayes strength of the prior (K = alpha + beta of a Beta-Binomial), method of moments (Kleinman, 1973)."""
    d, n = np.asarray(deserters, float), np.asarray(students, float)
    ok = np.isfinite(d) & np.isfinite(n) & (n > 0)
    d, n = d[ok], n[ok]
    big_n = len(d)
    if big_n < 3 or n.sum() - big_n <= 0:
        return np.nan
    m = d.sum() / n.sum()
    if not 0 < m < 1:
        return np.nan
    s = np.sum(n * (d / n - m) ** 2)
    rho = float(np.clip((s / (m * (1 - m)) - (big_n - 1)) / (n.sum() - big_n), 1e-6, 0.99))
    return (1 - rho) / rho


def load_panel(nombre: str) -> pd.DataFrame:
    """Final panel plus the students and deserters of the same sede in t-3, joined by calendar year."""
    df = pd.read_csv(
        IN_DIR / f"panel_final_{nombre}.csv", dtype={COL_SEDE: str, COL_MPIO: str}, low_memory=False
    )
    for col in (COL_PESO, COL_MPIO):
        if col not in df.columns:
            raise KeyError(f"{col} is missing from panel_final_{nombre}.csv: rerun the master panel and Phase 2.")
    hist = df[[COL_SEDE, COL_ANIO, COL_RATE, COL_PESO]].copy()
    hist[COL_ANIO] = hist[COL_ANIO] + HIST_LAG
    hist = hist.rename(columns={COL_RATE: "_rate_lag3", COL_PESO: "ESTUDIANTES_LAG3"})
    df = df.merge(hist, on=[COL_SEDE, COL_ANIO], how="left", validate="many_to_one")
    df["DESERTORES_LAG3"] = np.round(df["_rate_lag3"] * df["ESTUDIANTES_LAG3"])  # NaN where there is no record

    # The rebuilt history must match the DESERCION_LAG3 column of the panel
    both = df["_rate_lag3"].notna() & df[COL_HIST].notna()
    if both.any() and not np.allclose(df.loc[both, "_rate_lag3"], df.loc[both, COL_HIST], atol=1e-9):
        raise ValueError("The rate rebuilt for t-3 does not match DESERCION_LAG3: check the panel.")
    return df


def population(df: pd.DataFrame) -> pd.DataFrame:
    """Eligible rows with a Sit_Acad record (students known); deserters rebuilt from rate x students."""
    pop = df[(df["ELEGIBLE_MODELO"] == 1) & df[COL_PESO].gt(0) & df[COL_RATE].notna()].copy()
    pop["_d"] = np.round(pop[COL_RATE] * pop[COL_PESO])
    return pop


def add_shrunk_history(train: pd.DataFrame, test: pd.DataFrame, k: float) -> tuple:
    """
    DESERCION_LAG3_AJUSTADA = (deserters_t-3 + K * mean) / (students_t-3 + K). K and the mean come from the TRAIN
    years only; a sede without counts in t-3 gets the prior mean. The value is always > 0, so its log is defined.
    """
    m_all = train["_d"].sum() / train[COL_PESO].sum()
    out = []
    for frame in (train, test):
        frame = frame.copy()
        known = frame["ESTUDIANTES_LAG3"].gt(0)
        shrunk = (frame["DESERTORES_LAG3"] + k * m_all) / (frame["ESTUDIANTES_LAG3"] + k)
        frame[COL_HIST_ADJ] = np.where(known, shrunk, m_all)
        out.append(frame)
    return out[0], out[1]


def build_variants(base_features: list) -> list:
    """Feature sets and settings of the LightGBM variants (A, B, B2, C) for each capacity."""
    non_hist = [c for c in base_features if c != COL_HIST]
    all_feats = base_features + [COL_HIST_ADJ] + COLS_HIST_COUNTS
    mono = [1 if c in (COL_HIST, COL_HIST_ADJ) else 0 for c in all_feats]
    specs = []
    for t in TREE_GRID:
        specs += [
            dict(name=f"A_features_t{t}", feats=all_feats, offset=False, mono=None, trees=t),
            dict(name=f"B_offset_sin_hist_t{t}", feats=non_hist, offset=True, mono=None, trees=t),
            dict(name=f"B2_offset_con_hist_t{t}", feats=all_feats, offset=True, mono=None, trees=t),
            dict(name=f"C_offset_con_hist_mono_t{t}", feats=all_feats, offset=True, mono=mono, trees=t),
        ]
    return specs


def fit_history_glm(train: pd.DataFrame) -> PoissonRegressor:
    """Poisson GLM, rate ~ exp(b0 + b1 * log(shrunk history)), students as weight (same as an offset on log students)."""
    glm = PoissonRegressor(alpha=0.0, max_iter=500)
    glm.fit(np.log(train[[COL_HIST_ADJ]].to_numpy()), train[COL_RATE].to_numpy(), sample_weight=train[COL_PESO].to_numpy())
    return glm


def glm_linear_predictor(glm: PoissonRegressor, frame: pd.DataFrame) -> np.ndarray:
    """Linear predictor (log scale) of the history GLM: this is the offset passed to the GBM."""
    return glm.intercept_ + glm.coef_[0] * np.log(frame[COL_HIST_ADJ].to_numpy())


def fit_predict_lgbm(train, test, spec, off_train, off_test) -> tuple:
    """Poisson LightGBM (rate target, students as weight); with offset the GLM predictor is init_score."""
    params = dict(LGBM_PARAMS, n_estimators=spec["trees"])
    if spec["mono"] is not None:
        params["monotone_constraints"] = spec["mono"]
    mdl = lgb.LGBMRegressor(**params)
    fit_kwargs = dict(sample_weight=train[COL_PESO].to_numpy())
    if spec["offset"]:
        fit_kwargs["init_score"] = off_train
    mdl.fit(train[spec["feats"]], train[COL_RATE].to_numpy(), **fit_kwargs)
    raw = mdl.predict(test[spec["feats"]], raw_score=True)  # log scale, WITHOUT the offset
    if spec["offset"]:
        raw = raw + off_test
    return mdl, np.exp(raw)


def poisson_deviance(d: np.ndarray, mu: np.ndarray) -> float:
    """Mean Poisson deviance of the expected number of deserters (lower is better)."""
    mu = np.clip(mu, 1e-9, None)
    term = np.where(d > 0, d * np.log(np.where(d > 0, d, 1.0) / mu), 0.0)
    return float(np.mean(2 * (term - (d - mu))))


def capture_by_stratum(d, rate_hat, quintil, jitter, share=TOP_SHARE) -> list:
    """Inside each size quintile, % of the stratum's deserters in the top `share` of its sedes by predicted rate (random = 10%)."""
    caps = []
    for qn in range(5):
        idx = np.flatnonzero(quintil == qn)
        if idx.size == 0 or d[idx].sum() == 0:
            caps.append(np.nan)
            continue
        k = int(np.ceil(share * idx.size))
        top = idx[np.argsort(-(rate_hat[idx] + jitter[idx]))[:k]]
        caps.append(100 * d[top].sum() / d[idx].sum())
    return caps


def evaluate(test, rate_hat, label, anio, panel, quintil, n_pred, jitter) -> list:
    """Capture of deserters, fairness to small sedes, discrimination net of size, calibration and dispersion."""
    d = test["_d"].to_numpy()
    mu = rate_hat * test[COL_PESO].to_numpy()
    intra = capture_by_stratum(d, rate_hat, quintil, jitter)
    top_mu = mu >= np.quantile(mu, 0.90)
    pearson = float(np.mean((d - mu) ** 2 / np.clip(mu, 1e-2, None)))  # approximate: mu floored at 0.01
    filas = []
    for alpha in ALPHAS:
        score = rate_hat * n_pred ** alpha + jitter
        top = np.zeros(len(d), bool)
        top[np.argsort(-score)[: int(np.ceil(TOP_SHARE * len(d)))]] = True
        filas.append({
            "PANEL": panel, "ANIO_TEST": anio, "CONJUNTO": label, "ALPHA": alpha,
            "CAPTURA_DESERTORES": 100 * d[top].sum() / d.sum() if d.sum() else np.nan,
            "CAPTURA_Q1_PEQUENAS": 100 * float(d[top & (quintil == 0)].sum() / max(d[quintil == 0].sum(), 1)),
            "SPEARMAN": spearmanr(score, d)[0],
            "CALIBRACION": float(mu.sum() / d.sum()) if d.sum() else np.nan,
            "CALIB_TOP10_MU": float(d[top_mu].sum() / max(mu[top_mu].sum(), 1e-9)),
            "DEVIANZA_POISSON": poisson_deviance(d, mu),
            "PEARSON_APROX": pearson,
            "CAPTURA_DENTRO_ESTRATO_PROM": float(np.nanmean(intra)),
            "CAPTURA_DENTRO_Q1": intra[0], "CAPTURA_DENTRO_Q5": intra[4],
        })
    return filas


def core_metrics(d, rate_hat, peso, quintil, jitter) -> np.ndarray:
    """The three metrics used in the bootstrap: mean capture net of size, capture in Q5, Poisson deviance."""
    intra = capture_by_stratum(d, rate_hat, quintil, jitter)
    return np.array([np.nanmean(intra), intra[4], poisson_deviance(d, rate_hat * peso)])


def mean_core_metrics(store: dict, variants: list, counts=None) -> np.ndarray:
    """Core metrics averaged over the test years; `counts` = bootstrap multiplicity of each municipality (None = original data)."""
    out = np.zeros((len(variants), len(METRICS_BOOT)))
    for s in store.values():
        idx = np.arange(len(s["d"])) if counts is None else np.repeat(np.arange(len(s["d"])), counts[s["mpio_code"]])
        d, peso, quintil, jitter = s["d"][idx], s["peso"][idx], s["quintil"][idx], s["jitter"][idx]
        for j, v in enumerate(variants):
            out[j] += core_metrics(d, s["preds"][v][idx], peso, quintil, jitter)
    return out / len(store)


def bootstrap_pairs(store: dict, pairs: list, n_boot: int, panel: str) -> pd.DataFrame:
    """Paired bootstrap by municipality: difference (first - second) of the core metrics with a 95% percentile interval."""
    rng = np.random.default_rng(SEED)
    mpios = pd.unique(np.concatenate([s["mpio"] for s in store.values()]))
    code = pd.Series(np.arange(len(mpios)), index=mpios)
    for s in store.values():
        s["mpio_code"] = pd.Series(s["mpio"]).map(code).to_numpy()
    variants = sorted({v for pair in pairs for v in pair})
    pos = {v: i for i, v in enumerate(variants)}
    point = mean_core_metrics(store, variants)
    reps = np.zeros((n_boot, len(variants), len(METRICS_BOOT)))
    for b in range(n_boot):
        counts = np.bincount(rng.integers(0, len(mpios), len(mpios)), minlength=len(mpios))
        reps[b] = mean_core_metrics(store, variants, counts)
        if (b + 1) % 50 == 0:
            print(f"  bootstrap {panel}: {b + 1}/{n_boot}")
    filas = []
    for a, ref in pairs:
        for m, metric in enumerate(METRICS_BOOT):
            diff = reps[:, pos[a], m] - reps[:, pos[ref], m]
            better = (diff < 0) if metric in LOWER_IS_BETTER else (diff > 0)
            filas.append({
                "PANEL": panel, "VARIANTE": a, "REFERENCIA": ref, "METRICA": metric,
                "DIF_PUNTUAL": point[pos[a], m] - point[pos[ref], m],
                "IC95_BAJO": np.nanpercentile(diff, 2.5), "IC95_ALTO": np.nanpercentile(diff, 97.5),
                "P_MEJOR": float(np.nanmean(better)),
            })
    return pd.DataFrame(filas)


def family_shares(mdl, test: pd.DataFrame, spec: dict, glm) -> dict:
    """Mean |contribution| by family (TreeSHAP of the GBM; the offset history is added as the |deviation| of the GLM predictor)."""
    sample = test.sample(min(SHAP_MAX_ROWS, len(test)), random_state=SEED)
    contrib = mdl.predict(sample[spec["feats"]], pred_contrib=True)[:, :-1]
    mean_abs = dict(zip(spec["feats"], np.abs(contrib).mean(axis=0)))
    shares = {fam: float(sum(v for c, v in mean_abs.items() if c in cols)) for fam, cols in FAMILIAS.items()}
    if spec["offset"]:
        lin = glm_linear_predictor(glm, sample)
        shares["Historia propia"] += float(np.abs(lin - lin.mean()).mean())
    total = sum(shares.values())
    no_hist = total - shares["Historia propia"]
    out = {}
    for fam, v in shares.items():
        out[fam] = {"PCT_TOTAL": 100 * v / total, "PCT_SIN_HISTORIA": np.nan if fam == "Historia propia" else 100 * v / no_hist}
    return out


def to_markdown(df: pd.DataFrame, nd: int = 2) -> str:
    """Minimal markdown table (no extra dependency)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for r in df.itertuples(index=False):
        lines.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)


def save_report(tabla, boot, shap_tab, nombre) -> None:
    """Write csv and md; a locked OneDrive file must not abort the experiment."""
    try:
        AUDIT_DIR.mkdir(parents=True, exist_ok=True)
        stem = f"pre_modelo_experimento_offset_{nombre}"
        tabla.to_csv(AUDIT_DIR / f"{stem}.csv", index=False)
        boot.to_csv(AUDIT_DIR / f"pre_modelo_experimento_offset_bootstrap_{nombre}.csv", index=False)
        shap_tab.to_csv(AUDIT_DIR / f"pre_modelo_experimento_offset_shap_familias_{nombre}.csv", index=False)
        base = tabla[tabla["ALPHA"] == ALPHAS[0]]
        intra = base.groupby("CONJUNTO")[
            ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON",
             "CALIBRACION", "CALIB_TOP10_MU", "PEARSON_APROX"]
        ].mean().reset_index().sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False)
        por_anio = base.pivot_table(index="CONJUNTO", columns="ANIO_TEST", values="CAPTURA_DENTRO_ESTRATO_PROM").reset_index()
        por_anio.columns = [str(c) for c in por_anio.columns]
        listas = tabla.groupby(["CONJUNTO", "ALPHA"])[["CAPTURA_DESERTORES", "CAPTURA_Q1_PEQUENAS", "SPEARMAN"]].mean().reset_index()
        md = [
            f"# Experimento de offset - panel {nombre}",
            "\n## Discriminación dentro de cada estrato de tamaño, devianza y calibración (promedio de los años; azar = 10%)\n",
            "CALIBRACION ideal = 1; CALIB_TOP10_MU = observados / esperados en el 10% de mayor mu; PEARSON_APROX cercano a 1 = sin sobredispersión (mu con piso de 0,01).\n",
            to_markdown(intra),
            "\n## Captura dentro del estrato por año de prueba\n",
            to_markdown(por_anio),
            f"\n## Diferencias pareadas con IC 95% (bootstrap por municipio, {N_BOOT} réplicas; positivo = mejor salvo DEVIANZA_POISSON, donde negativo = mejor)\n",
            "P_MEJOR = proporción de réplicas en que la variante supera a la referencia.\n",
            to_markdown(boot.drop(columns=["PANEL"]), nd=3),
            "\n## Participación por familia (último año de prueba; % de la contribución absoluta media)\n",
            to_markdown(shap_tab.drop(columns=["PANEL"])),
            "\n## Lista del 10% de sedes por variante y alpha (promedio de los años)\n",
            to_markdown(listas),
        ]
        (AUDIT_DIR / f"{stem}.md").write_text("\n".join(md), encoding="utf-8")
    except OSError as err:
        print(f"[WARN] Could not write the report of {nombre}: {err}")


def run_panel(nombre: str, base_features: list) -> tuple:
    """Rolling-origin backtests for one panel: baselines + LightGBM variants A, B, B2, C (two capacities each)."""
    pop = population(load_panel(nombre))
    specs = build_variants(base_features)
    rng = np.random.default_rng(SEED)
    filas, store, shap_rows = [], {}, []
    for anio in TEST_YEARS:
        train, test = pop[pop[COL_ANIO] < anio], pop[pop[COL_ANIO] == anio]
        if train.empty or test.empty:
            print(f"[WARN] {nombre} {anio}: no train or test rows, skipped.")
            continue
        k_eb = estimate_k_empirical_bayes(train["_d"], train[COL_PESO])
        print(f"{nombre} | test {anio} | train rows {len(train):,} | test rows {len(test):,} | K empirical Bayes = {k_eb:.1f}")
        train, test = add_shrunk_history(train, test, k_eb)
        glm = fit_history_glm(train)
        off_train, off_test = glm_linear_predictor(glm, train), glm_linear_predictor(glm, test)
        print(f"  history GLM: intercept = {glm.intercept_:.3f}, slope on log(shrunk history) = {glm.coef_[0]:.3f}")

        n_pred = test[COL_SIZE].fillna(test[COL_SIZE].median()).clip(lower=1).to_numpy()
        quintil = pd.qcut(pd.Series(n_pred).rank(method="first"), 5, labels=False).to_numpy()
        jitter = rng.random(len(test)) * 1e-12
        store[anio] = dict(
            d=test["_d"].to_numpy(), peso=test[COL_PESO].to_numpy(), quintil=quintil, jitter=jitter,
            mpio=test[COL_MPIO].astype(str).to_numpy(), preds={},
        )

        def register(label: str, rate_hat: np.ndarray) -> None:
            filas.extend(evaluate(test, rate_hat, label, anio, nombre, quintil, n_pred, jitter))
            store[anio]["preds"][label] = rate_hat

        # Baselines
        register("B0b_historia_ajustada", test[COL_HIST_ADJ].to_numpy())
        register("GLM_solo_historia", np.exp(off_test))
        hgb = HistGradientBoostingRegressor(loss="poisson", **HGB_PARAMS)  # reproduction of the previous M4_BASE
        hgb.fit(train[base_features], train[COL_RATE], sample_weight=train[COL_PESO])
        register("M4_BASE_HGB", hgb.predict(test[base_features]))

        # LightGBM variants
        for spec in specs:
            mdl, rate_hat = fit_predict_lgbm(train, test, spec, off_train, off_test)
            register(spec["name"], rate_hat)
            if anio == TEST_YEARS[-1] and spec["trees"] == TREE_GRID[-1] and not spec["name"].startswith("C_"):
                for fam, vals in family_shares(mdl, test, spec, glm).items():
                    shap_rows.append({"PANEL": nombre, "VARIANTE": spec["name"], "FAMILIA": fam, **vals})
    tabla = pd.DataFrame(filas)
    ref = f"A_features_t{TREE_GRID[-1]}"
    names = [s["name"] for s in specs] + ["B0b_historia_ajustada", "GLM_solo_historia", "M4_BASE_HGB"]
    pairs = [(v, ref) for v in names if v != ref]
    pairs += [(f"B_offset_sin_hist_t{t}", f"B2_offset_con_hist_t{t}") for t in TREE_GRID]   # cost of leaving the lag out
    pairs += [(f"{p}_t{TREE_GRID[-1]}", "GLM_solo_historia") for p in ("B_offset_sin_hist", "B2_offset_con_hist")]  # added value of the GBM
    return tabla, store, pairs, pd.DataFrame(shap_rows)


# =============================================================================
# 3. EXECUTION
# =============================================================================

def main() -> None:
    for nombre, features in PANELES.items():
        tabla, store, pairs, shap_tab = run_panel(nombre, features)
        boot = bootstrap_pairs(store, pairs, N_BOOT, nombre)
        save_report(tabla, boot, shap_tab, nombre)
        print(f"\n=== {nombre}: discrimination net of size (mean over test years; random = 10%) ===")
        print(
            tabla[tabla["ALPHA"] == ALPHAS[0]].groupby("CONJUNTO")[
                ["CAPTURA_DENTRO_ESTRATO_PROM", "CAPTURA_DENTRO_Q1", "CAPTURA_DENTRO_Q5", "DEVIANZA_POISSON", "CALIBRACION"]
            ].mean().round(2).sort_values("CAPTURA_DENTRO_ESTRATO_PROM", ascending=False).to_string()
        )
        print(f"\n=== {nombre}: paired differences with 95% bootstrap interval (by municipality) ===")
        print(boot.drop(columns=["PANEL"]).round(3).to_string(index=False))
        print(f"\n=== {nombre}: share by family, last test year ===")
        print(shap_tab.pivot_table(index="FAMILIA", columns="VARIANTE", values="PCT_TOTAL").round(1).to_string())


if __name__ == "__main__":
    main()


ofrecen11 | test 2021 | train rows 11,906 | test rows 11,984 | K empirical Bayes = 13.4
  history GLM: intercept = -2.114, slope on log(shrunk history) = 0.341
ofrecen11 | test 2022 | train rows 23,890 | test rows 12,146 | K empirical Bayes = 14.3
  history GLM: intercept = -1.919, slope on log(shrunk history) = 0.370
ofrecen11 | test 2023 | train rows 36,036 | test rows 11,815 | K empirical Bayes = 14.9
  history GLM: intercept = -1.848, slope on log(shrunk history) = 0.388
  bootstrap ofrecen11: 50/200
  bootstrap ofrecen11: 100/200
  bootstrap ofrecen11: 150/200
  bootstrap ofrecen11: 200/200

=== ofrecen11: discrimination net of size (mean over test years; random = 10%) ===
                             CAPTURA_DENTRO_ESTRATO_PROM  CAPTURA_DENTRO_Q1  CAPTURA_DENTRO_Q5  DEVIANZA_POISSON  CALIBRACION
CONJUNTO                                                                                                                     
B0b_historia_ajustada                              23.78     